# Figure 7: DNN maximal-feasible charging

This is the canonical notebook entry point for Figure 7. Running both code cells regenerates `paper_figures/figure_07_nonlinear_maximal_feasible.pdf` from the matching JSON settings file.

The numerical solver is deliberately kept in `paper_figures/generate_figure_07.py`, which this notebook calls directly. This gives the notebook and command-line workflow one identical implementation rather than two copies that could diverge.

The configured case uses $H(t)=\bar c(t)-c_s(t)$, a 65 s horizon, $D_{\mathrm{ref}}=D(c=1)$, $j_{\max}=60$, $\bar c:1\to0.6$, $\beta\in\{0,0.5,1\}$, and $H_{\mathrm{con}}=8H_{\mathrm{peak}}/9$.


In [ ]:
from __future__ import annotations

import json
import sys
from math import isclose
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the repository root whether Jupyter starts here or above it."""
    for candidate in (start, *start.parents):
        if (candidate / 'paper_figures' / 'figure_07_settings.json').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the IGA Codes folder.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SETTINGS_PATH = PROJECT_ROOT / 'paper_figures' / 'figure_07_settings.json'
OUTPUT_PATH = PROJECT_ROOT / 'paper_figures' / 'figure_07_nonlinear_maximal_feasible.pdf'

with SETTINGS_PATH.open(encoding='utf-8') as stream:
    settings = json.load(stream)

assert settings['discretization']['horizon_s'] == 65.0
assert isclose(
    settings['operating_conditions']['constraint_fraction_of_linear_full_current_peak'],
    8.0 / 9.0,
)
assert settings['operating_conditions']['beta_values'] == [0.0, 0.5, 1.0]
print(f'Settings: {SETTINGS_PATH}')
print(f'Output:   {OUTPUT_PATH}')


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from paper_figures.generate_figure_07 import reproduce

summary = reproduce(SETTINGS_PATH)
assert summary['output'].resolve() == OUTPUT_PATH.resolve()
assert OUTPUT_PATH.is_file()

print(f"H_peak = {summary['h_max']:.12g} at {summary['h_max_time_s']:.9g} s")
print(f"H_con = {summary['h_con']:.12g} ({summary['constraint_fraction']:.6g} H_peak)")
for beta, result in summary['results'].items():
    print(
        f"beta={beta:g}: entry={result['entry_time_s']:.9g} s, "
        f"target={result['target_time_s']:.9g} s, "
        f"max(H-H_con)={result['maximum_violation']:.3e}"
    )
print(f'Wrote {OUTPUT_PATH}')
